In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

df_normalized = pd.read_csv('anova_chi_normalized.csv') # change csv file name
target_normalized = df_normalized['AgreeSubsequentBooster']


features_normalized = df_normalized.drop('AgreeSubsequentBooster', axis=1)

x_train_normalized, x_test_normalized, y_train_normalized, y_test_normalized = train_test_split(features_normalized,
                                                                                                target_normalized,
                                                                                                test_size=0.2,
                                                                                                random_state=0)

In [2]:
anova_columns = [
    'Age',
    'EmploymentYears',
    'A_mean',
    'C_mean',
    'D_mean',
    'E_mean'
]

chi2_columns = [
    'Education',
    'SideEffectsAffectView'
]

# Extract and save ANOVA columns from the training set
anova_train = x_train_normalized[anova_columns]

# Extract and save Chi-Squared columns from the training set
chi2_train = x_train_normalized[chi2_columns]

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, chi2

# ----------------------------
# Helper: drop constant columns
# ----------------------------
def drop_constant_columns(X):
    # Keep columns that have more than 1 unique value
    return X.loc[:, X.nunique(dropna=False) > 1]

# ----------------------------
# 1) ANOVA on numeric features
# ----------------------------
anova_train_clean = drop_constant_columns(anova_train)

anova_selector = SelectKBest(score_func=f_classif, k='all')
anova_selector.fit(anova_train_clean, y_train_normalized)

anova_scores = anova_selector.scores_
anova_pvals  = anova_selector.pvalues_

feature_scores_anova = pd.DataFrame({
    "Feature": anova_train_clean.columns,
    "Score": anova_scores,          # F-statistic
    "P-value": anova_pvals,
    "Method": "ANOVA"
})

# ----------------------------
# 2) Chi-square on categorical/non-negative features
# ----------------------------
chi2_train_clean = drop_constant_columns(chi2_train)

# IMPORTANT: chi2 requires non-negative values
if (chi2_train_clean.values < 0).any():
    raise ValueError("Chi-square requires non-negative features. Check your preprocessing for chi2 columns.")

chi_selector = SelectKBest(score_func=chi2, k='all')
chi_selector.fit(chi2_train_clean, y_train_normalized)

chi_scores = chi_selector.scores_     # chi-square statistic
chi_pvals  = chi_selector.pvalues_

feature_scores_chi = pd.DataFrame({
    "Feature": chi2_train_clean.columns,
    "Score": chi_scores,
    "P-value": chi_pvals,
    "Method": "Chi2"
})

# ----------------------------
# 3) Combine + sort baseline ranking
# ----------------------------
feature_scores = pd.concat([feature_scores_anova, feature_scores_chi], ignore_index=True)

# Drop any NaNs just in case
feature_scores = feature_scores.dropna(subset=["P-value"])

# Sort: smallest p-value = most important
feature_scores_sorted = feature_scores.sort_values(by="P-value", ascending=False).reset_index(drop=True)

print(feature_scores_sorted.head(30))

# Save baseline ranking
feature_scores_sorted.to_csv("p_values_for_base.csv", index=False)

                 Feature      Score   P-value Method
0                    Age   4.341839  0.038807  ANOVA
1              Education   5.062757  0.024445   Chi2
2        EmploymentYears   9.016271  0.003115  ANOVA
3                 D_mean  11.085383  0.001084  ANOVA
4                 E_mean  12.942599  0.000430  ANOVA
5  SideEffectsAffectView  13.275127  0.000269   Chi2
6                 A_mean  21.085981  0.000009  ANOVA
7                 C_mean  25.738069  0.000001  ANOVA


In [4]:
feature_scores_sorted.to_csv('p_values_for_base.csv')

In [5]:
# boosting shap values

import shap
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier

# Train model
model_boosting = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model_boosting.fit(x_train_normalized, y_train_normalized)

# SHAP (TreeExplainer)
explainer = shap.TreeExplainer(model_boosting)

# Use the call-style API so we can handle Explanation objects cleanly
sv = explainer(x_test_normalized)

# sv can be:
# - shap.Explanation (common in newer SHAP)
# - list of arrays (older style)
# - ndarray
if hasattr(sv, "values"):   # Explanation
    values = sv.values
else:
    values = sv

# Now handle shapes:
# Case A: values is list [class0, class1]
if isinstance(values, list):
    shap_pos = values[1]   # class 1 matrix (n_samples, n_features)

# Case B: values is ndarray
else:
    # If 3D: (n_samples, n_features, n_classes) -> take class 1
    if values.ndim == 3:
        shap_pos = values[:, :, 1]
    # If 2D: already (n_samples, n_features)
    elif values.ndim == 2:
        shap_pos = values
    else:
        raise ValueError(f"Unexpected SHAP values shape: {values.shape}")

# Finally: mean absolute SHAP per feature -> 1D (n_features,)
shap_importance = np.abs(shap_pos).mean(axis=0)


print("shap_pos shape:", np.shape(shap_pos))
print("shap_importance shape:", np.shape(shap_importance))
print("num features:", len(x_test_normalized.columns))

# Build ranking boosting
shap_rank_boosting = pd.DataFrame({
    "Feature": x_test_normalized.columns,
    "SHAP_Importance": shap_importance
}).sort_values("SHAP_Importance", ascending=False)

shap_rank_boosting.to_csv("Booster_boosting_shap_output.csv", index=False)
print(shap_rank_boosting.head(20))

shap_pos shape: (40, 8)
shap_importance shape: (8,)
num features: 8
                 Feature  SHAP_Importance
0                    Age         0.989803
3                 C_mean         0.675167
7  SideEffectsAffectView         0.638063
2                 A_mean         0.492001
6              Education         0.434212
1        EmploymentYears         0.385540
4                 D_mean         0.233247
5                 E_mean         0.224610


In [6]:
from sklearn.naive_bayes import GaussianNB
# gnb shap values

import shap
import numpy as np
import pandas as pd

model_gnb = GaussianNB()
model_gnb.fit(x_train_normalized, y_train_normalized)

background = shap.sample(x_train_normalized, 50, random_state=0)
X_explain = shap.sample(x_test_normalized, 200, random_state=0)

def model_predict_proba(X):
    X = pd.DataFrame(X, columns=x_train_normalized.columns)
    return model_gnb.predict_proba(X)

explainer = shap.KernelExplainer(model_predict_proba, background)

sv = explainer.shap_values(X_explain, nsamples=200)

if isinstance(sv, list):
    shap_pos = sv[1]                         # (n_samples, n_features)
else:
    # If it’s 3D (n_samples, n_features, n_classes), take class 1
    shap_pos = sv[:, :, 1] if sv.ndim == 3 else sv

shap_importance = np.abs(shap_pos).mean(axis=0).ravel()

print("Columns in X_explain:", X_explain.shape[1])
print("Length shap_importance:", len(shap_importance))

shap_rank_gnb = pd.DataFrame({
    "Feature": X_explain.columns,
    "SHAP_Importance": shap_importance
}).sort_values("SHAP_Importance", ascending=False)

shap_rank_gnb.to_csv("Booster_gnb_shap_output.csv", index=False)
print(shap_rank_gnb.head(20))

  0%|          | 0/40 [00:00<?, ?it/s]

Columns in X_explain: 8
Length shap_importance: 8
                 Feature  SHAP_Importance
3                 C_mean         0.094180
1        EmploymentYears         0.090610
2                 A_mean         0.086281
6              Education         0.068579
7  SideEffectsAffectView         0.067824
4                 D_mean         0.051327
5                 E_mean         0.044799
0                    Age         0.039955
